In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
from ipywidgets import FloatSlider, IntSlider, ToggleButtons, Play, jslink, HBox, VBox, HTML, Layout, interactive
from IPython.display import display


# ============================================================
# OVERFLOW LIMIT CYCLES AND RECOVERY IN THE STATE PLANE
# ============================================================
#
# This notebook studies large-scale overflow nonlinearities in a
# second-order recursive system.
#
# It is intentionally separated from the previous notebooks on
# small-scale quantization limit cycles.
#
#
# ============================================================
# SECOND-ORDER STATE-SPACE MODEL
# ============================================================
#
# Consider the homogeneous recursion
#
#       y[n+2] = alpha y[n+1] + beta y[n].
#
# Define the state
#
#       q[n] = [q1[n], q2[n]]^T
#            = [y[n], y[n+1]]^T.
#
# Then
#
#       q[n+1] = A q[n]
#
# with
#
#                  [ 0      1    ]
#       A =        [             ].
#                  [ beta   alpha ]
#
#
# ============================================================
# STABILITY
# ============================================================
#
# The characteristic polynomial is
#
#       lambda^2 - alpha lambda - beta = 0.
#
# Linear stability requires both poles to lie strictly inside the
# unit circle.
#
# In the (beta, alpha) parameter plane the stable region is the
# triangle with vertices
#
#       (beta, alpha) = ( 1,  0)
#                       (-1,  2)
#                       (-1, -2).
#
#
# ============================================================
# STRONGER NO-OVERFLOW-OSCILLATION CONDITION
# ============================================================
#
# The stronger condition
#
#       |alpha| + |beta| < 1
#
# defines a diamond inside the linear stability triangle.
#
# Therefore,
#
#       linear stability
#
# does NOT automatically imply
#
#       absence of nonlinear overflow oscillations.
#
#
# ============================================================
# STATE RANGE
# ============================================================
#
# The finite state range is represented by the square
#
#       -1 <= q1 <= 1
#       -1 <= q2 <= 1.
#
# Whenever the next state leaves this square, an overflow recovery
# rule is applied componentwise.
#
#
# ============================================================
# OVERFLOW RECOVERY RULES
# ============================================================
#
# Saturation
# ----------
#
#       q <- clip(q, -1, 1)
#
# An overflowing component is forced to the nearest boundary.
#
#
# Zeroing
# -------
#
#       q <- 0
#
# for each component that leaves the admissible range.
#
#
# Two's-complement wrap
# ---------------------
#
# The value is wrapped modulo the full-scale interval of length 2:
#
#       q <- ((q + 1) mod 2) - 1.
#
# Examples:
#
#       1.2  -> -0.8
#      -1.2  ->  0.8
#
#
# ============================================================
# IMPORTANT MODELING CHOICE
# ============================================================
#
# This notebook isolates OVERFLOW nonlinearity.
#
# No small-scale fixed-point quantization is applied to ordinary
# arithmetic operations.
#
# Therefore, any nonlinear behavior shown here is caused by the
# overflow recovery mechanism rather than ordinary rounding or
# truncation.
#
#
# ============================================================
# TWO INTERACTIVE MODES
# ============================================================
#
# STATE PLANE
# -----------
#
# This mode follows the time evolution of the state vector.
#
# It shows:
#
#       - the square -1 <= q1,q2 <= 1,
#       - the uncorrected linear trajectory,
#       - the overflow-corrected trajectory,
#       - raw overflowing states,
#       - the current state,
#       - state components versus iteration,
#       - overflow and repetition information.
#
# The trajectory can be advanced manually with the Time step slider
# or automatically with the Play control.
#
#
# PARAMETER PLANE
# ---------------
#
# This mode does not display time evolution.
#
# It places the selected system in the (beta, alpha) parameter plane
# and compares two regions:
#
#       1. the linear-stability triangle,
#       2. the stronger |alpha| + |beta| < 1 region.
#
# The purpose is to show visually that a system may be linearly
# stable while not satisfying the stronger condition associated
# with the absence of overflow oscillations.
#
# ============================================================


# ------------------------------------------------------------
# Pole calculation
# ------------------------------------------------------------

def calculate_poles(alpha, beta):

    return np.roots([1.0, -alpha, -beta])


# ------------------------------------------------------------
# Linear stability
# ------------------------------------------------------------

def is_linearly_stable(alpha, beta):

    poles = calculate_poles(alpha, beta)

    return np.all(np.abs(poles) < 1.0 - 1e-12)


# ------------------------------------------------------------
# Stronger no-overflow-oscillation condition
# ------------------------------------------------------------

def satisfies_safe_condition(alpha, beta):

    return abs(alpha) + abs(beta) < 1.0 - 1e-12


# ------------------------------------------------------------
# State matrix
# ------------------------------------------------------------

def state_matrix(alpha, beta):

    return np.array([[0.0, 1.0], [beta, alpha]], dtype=float)


# ------------------------------------------------------------
# Overflow detection
# ------------------------------------------------------------

def state_overflows(q):

    return np.any(np.abs(q) > 1.0 + 1e-12)


# ------------------------------------------------------------
# Saturation correction
# ------------------------------------------------------------

def saturation_correction(q):

    return np.clip(q, -1.0, 1.0)


# ------------------------------------------------------------
# Zeroing correction
# ------------------------------------------------------------

def zeroing_correction(q):

    q_corrected = np.asarray(q, dtype=float).copy()

    q_corrected[np.abs(q_corrected) > 1.0] = 0.0

    return q_corrected


# ------------------------------------------------------------
# Two's-complement wrap correction
# ------------------------------------------------------------

def wrap_component(x):

    return ((x + 1.0) % 2.0) - 1.0


def twos_complement_correction(q):

    return np.array([wrap_component(q[0]), wrap_component(q[1])], dtype=float)


# ------------------------------------------------------------
# Select overflow recovery mechanism
# ------------------------------------------------------------

def apply_recovery(q, recovery):

    if recovery == 'Saturation':

        return saturation_correction(q)

    if recovery == 'Zeroing':

        return zeroing_correction(q)

    return twos_complement_correction(q)


# ------------------------------------------------------------
# Simulate the uncorrected linear trajectory
# ------------------------------------------------------------

def simulate_linear(alpha, beta, q10, q20, iterations):

    A = state_matrix(alpha, beta)

    q = np.zeros((iterations + 1, 2), dtype=float)

    q[0, :] = [q10, q20]

    for n in range(iterations):

        q[n + 1, :] = A @ q[n, :]

    return q


# ------------------------------------------------------------
# Simulate the overflow-corrected trajectory
# ------------------------------------------------------------

def simulate_corrected(alpha, beta, q10, q20, iterations, recovery):

    A = state_matrix(alpha, beta)

    q = np.zeros((iterations + 1, 2), dtype=float)

    q[0, :] = [q10, q20]

    overflow_events = []

    raw_states = []

    for n in range(iterations):

        raw_next = A @ q[n, :]

        raw_states.append(raw_next.copy())

        if state_overflows(raw_next):

            overflow_events.append(n + 1)

            q[n + 1, :] = apply_recovery(raw_next, recovery)

        else:

            q[n + 1, :] = raw_next

    return q, np.asarray(raw_states), overflow_events


# ------------------------------------------------------------
# Detect a repeated corrected state approximately
# ------------------------------------------------------------

def detect_repeated_state(q, decimals=10):

    visited = {}

    for n in range(len(q)):

        key = tuple(np.round(q[n, :], decimals=decimals))

        if key in visited:

            first = visited[key]

            second = n

            period = second - first

            return first, second, period

        visited[key] = n

    return None, None, None


# ============================================================
# CSS
# ============================================================

style_html = HTML("""
<style>

.olc-root {
    width: 960px;
    max-width: 960px;
    font-family: Arial, sans-serif;
}

.olc-header {
    background: #2f3742;
    color: white;
    padding: 8px 14px;
    border-radius: 7px 7px 0 0;
    font-size: 19px;
    font-weight: bold;
}

.olc-intro {
    background: #f5f7f9;
    border: 1px solid #d4d9df;
    border-top: none;
    padding: 7px 12px;
    border-radius: 0 0 7px 7px;
    font-size: 12px;
    line-height: 1.45;
    margin-bottom: 6px;
}

.olc-accent {
    font-weight: bold;
    color: #384f67;
}

.olc-controls-title {
    font-size: 12.5px;
    font-weight: bold;
    margin: 0 0 3px 3px;
    color: #303845;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea {
    overflow-x: visible !important;
    max-width: none !important;
}

</style>
""")


# ============================================================
# VISIBLE DOCUMENTATION
# ============================================================

header_html = HTML("""
<div class="olc-root">

    <div class="olc-header">
        Overflow Limit Cycles and Recovery in the State Plane
    </div>

    <div class="olc-intro">

        <span class="olc-accent">State plane:</span>
        follows the time evolution of the state vector inside the square
        -1 &le; q1 &le; 1, -1 &le; q2 &le; 1 and shows how
        Saturation, Zeroing, or Two's-complement wrap modifies the
        trajectory when overflow occurs.

        <br>

        <span class="olc-accent">Parameter plane:</span>
        places the selected system in the (beta, alpha) plane and shows
        both the linear-stability triangle and the stronger
        |alpha| + |beta| &lt; 1 no-overflow-oscillation region.

    </div>

</div>
""")


# ============================================================
# MAIN INTERACTIVE PLOTTING FUNCTION
# ============================================================

def plot_overflow_lab(view='State-plane experiment', alpha=1.50, beta=-0.80, q10=-0.90, q20=0.90, recovery='Saturation', iterations=30, time_step=1):

    poles = calculate_poles(alpha, beta)

    stable = is_linearly_stable(alpha, beta)

    safe = satisfies_safe_condition(alpha, beta)


    # ========================================================
    # MODE 1 — STATE-PLANE EXPERIMENT
    # ========================================================

    if view == 'State-plane experiment':

        q_linear_full = simulate_linear(alpha, beta, q10, q20, iterations)

        q_corrected_full, q_raw_full, overflow_events = simulate_corrected(alpha, beta, q10, q20, iterations, recovery)

        current_step = int(np.clip(time_step, 1, iterations))

        q_linear = q_linear_full[:current_step + 1, :]

        q_corrected = q_corrected_full[:current_step + 1, :]

        visible_overflows = [n for n in overflow_events if n <= current_step]

        first_repeat, second_repeat, period = detect_repeated_state(q_corrected)


        # ----------------------------------------------------
        # Figure layout
        # ----------------------------------------------------

        fig = plt.figure(figsize=(13.2, 6.8))

        grid = fig.add_gridspec(2, 2, width_ratios=[1.35, 1.0], height_ratios=[1.0, 0.92], wspace=0.30, hspace=0.48)

        ax_state = fig.add_subplot(grid[:, 0])

        ax_components = fig.add_subplot(grid[0, 1])

        ax_monitor = fig.add_subplot(grid[1, 1])


        # ====================================================
        # LARGE STATE-PLANE PANEL
        # ====================================================

        square = Rectangle((-1.0, -1.0), 2.0, 2.0, fill=False, linewidth=2.0, linestyle='-', edgecolor='0.25', label='State range')

        ax_state.add_patch(square)


        # ----------------------------------------------------
        # Reference state-plane grid
        # ----------------------------------------------------

        for value in np.arange(-1.0, 1.01, 0.2):

            ax_state.axvline(value, linewidth=0.45, alpha=0.16)

            ax_state.axhline(value, linewidth=0.45, alpha=0.16)


        # ----------------------------------------------------
        # Uncorrected linear trajectory
        # ----------------------------------------------------

        ax_state.plot(
            q_linear[:, 0],
            q_linear[:, 1],
            '--o',
            color='tab:blue',
            linewidth=1.5,
            markersize=3.7,
            label='Uncorrected linear trajectory'
        )


        # ----------------------------------------------------
        # Overflow-corrected trajectory
        # ----------------------------------------------------

        ax_state.plot(
            q_corrected[:, 0],
            q_corrected[:, 1],
            'o-',
            color='tab:orange',
            linewidth=1.9,
            markersize=5.0,
            label=f'{recovery} trajectory'
        )


        # ----------------------------------------------------
        # Current corrected state
        # ----------------------------------------------------

        ax_state.plot(
            q_corrected[-1, 0],
            q_corrected[-1, 1],
            'o',
            color='tab:green',
            markersize=11,
            markerfacecolor='none',
            markeredgewidth=2.2,
            label='Current state'
        )


        # ----------------------------------------------------
        # Initial state
        # ----------------------------------------------------

        ax_state.plot(
            q_corrected_full[0, 0],
            q_corrected_full[0, 1],
            'D',
            color='tab:purple',
            markersize=7,
            label='Initial state'
        )


        # ----------------------------------------------------
        # Raw overflowing states
        # ----------------------------------------------------

        overflow_label_used = False


        for event in visible_overflows:

            raw_state = q_raw_full[event - 1, :]

            label = 'Raw overflowing state' if not overflow_label_used else None


            ax_state.plot(
                raw_state[0],
                raw_state[1],
                'X',
                color='tab:red',
                markersize=9,
                markeredgewidth=1.4,
                label=label
            )


            ax_state.plot(
                [q_corrected_full[event - 1, 0], raw_state[0]],
                [q_corrected_full[event - 1, 1], raw_state[1]],
                ':',
                color='tab:red',
                linewidth=1.3
            )


            ax_state.plot(
                [raw_state[0], q_corrected_full[event, 0]],
                [raw_state[1], q_corrected_full[event, 1]],
                ':',
                color='tab:green',
                linewidth=1.3
            )


            overflow_label_used = True


        # ----------------------------------------------------
        # Origin
        # ----------------------------------------------------

        ax_state.plot(
            0.0,
            0.0,
            '+',
            color='0.3',
            markersize=11,
            markeredgewidth=1.8
        )


        # ----------------------------------------------------
        # Dynamic plot range
        # ----------------------------------------------------

        visible_x = np.concatenate((
            q_linear[:, 0],
            q_corrected[:, 0],
            np.array([-1.0, 1.0])
        ))


        visible_y = np.concatenate((
            q_linear[:, 1],
            q_corrected[:, 1],
            np.array([-1.0, 1.0])
        ))


        if len(visible_overflows) > 0:

            raw_visible = np.asarray([
                q_raw_full[n - 1, :]
                for n in visible_overflows
            ])


            visible_x = np.concatenate((
                visible_x,
                raw_visible[:, 0]
            ))


            visible_y = np.concatenate((
                visible_y,
                raw_visible[:, 1]
            ))


        max_extent = max(
            np.max(np.abs(visible_x)),
            np.max(np.abs(visible_y)),
            1.0
        )


        limit = 1.12 * max_extent


        ax_state.set_xlim(
            -limit,
            limit
        )


        ax_state.set_ylim(
            -limit,
            limit
        )


        ax_state.set_aspect(
            'equal',
            adjustable='box'
        )


        ax_state.set_xlabel(
            r'$q_1[n]$',
            fontsize=12,
            labelpad=8
        )


        ax_state.set_ylabel(
            r'$q_2[n]$',
            fontsize=12,
            labelpad=8
        )


        ax_state.set_title(
            f'State-Plane Evolution — Time Step n = {current_step}',
            fontsize=13
        )


        ax_state.tick_params(
            labelsize=9
        )


        ax_state.grid(
            True,
            linestyle=':',
            alpha=0.25
        )


        ax_state.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.10),
            ncol=2,
            frameon=False,
            fontsize=8.8,
            columnspacing=1.3
        )


        # ====================================================
        # CORRECTED STATE COMPONENTS
        # ====================================================

        n_values = np.arange(
            current_step + 1
        )


        ax_components.plot(
            n_values,
            q_corrected[:, 0],
            'o-',
            linewidth=1.4,
            markersize=4.0,
            label=r'Corrected $q_1[n]$'
        )


        ax_components.plot(
            n_values,
            q_corrected[:, 1],
            's-',
            linewidth=1.4,
            markersize=3.8,
            label=r'Corrected $q_2[n]$'
        )


        ax_components.axhline(
            1.0,
            linestyle=':',
            linewidth=1.1
        )


        ax_components.axhline(
            -1.0,
            linestyle=':',
            linewidth=1.1
        )


        ax_components.set_xlim(
            0,
            iterations
        )


        ax_components.set_ylim(
            -1.12,
            1.12
        )


        ax_components.set_xlabel(
            'Iteration n',
            labelpad=7
        )


        ax_components.set_ylabel(
            'State component'
        )


        ax_components.set_title(
            'Corrected State Components',
            fontsize=11
        )


        ax_components.grid(
            True,
            linestyle=':',
            alpha=0.28
        )


        ax_components.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.22),
            ncol=2,
            frameon=False,
            fontsize=8.8
        )


        # ====================================================
        # MONITOR
        # ====================================================

        ax_monitor.axis(
            'off'
        )


        pole_text = (
            f'p1 = {poles[0].real:+.4f}{poles[0].imag:+.4f}j\n'
            f'p2 = {poles[1].real:+.4f}{poles[1].imag:+.4f}j'
        )


        if stable and safe:

            interpretation = (
                'LINEARLY STABLE\n'
                'SAFE CONDITION SATISFIED'
            )


        elif stable and not safe:

            interpretation = (
                'LINEARLY STABLE\n'
                'OVERFLOW OSCILLATIONS POSSIBLE'
            )


        else:

            interpretation = (
                'LINEARLY UNSTABLE'
            )


        if len(visible_overflows) == 0:

            overflow_text = (
                'No overflow event yet.'
            )


        else:

            overflow_text = (
                f'Overflow events shown: {len(visible_overflows)}\n'
                f'First overflow: n = {visible_overflows[0]}'
            )


        if period is None:

            cycle_text = (
                'No repeated corrected state detected.'
            )


        elif period == 1:

            cycle_text = (
                'Repeated state detected\n'
                'Period = 1'
            )


        else:

            cycle_text = (
                f'Repeated corrected state detected\n'
                f'Period = {period}'
            )


        left_text = (
            f'SYSTEM\n'
            f'────────────────────\n'
            f'alpha      : {alpha:+.3f}\n'
            f'beta       : {beta:+.3f}\n'
            f'|a|+|b|    : {abs(alpha) + abs(beta):.3f}\n'
            f'Recovery   : {recovery}\n'
            f'Time n     : {current_step}\n\n'
            f'POLES\n'
            f'{pole_text}'
        )


        right_text = (
            f'STABILITY / OVERFLOW STATUS\n'
            f'──────────────────────────\n'
            f'{interpretation}\n\n'
            f'{overflow_text}\n\n'
            f'{cycle_text}'
        )


        # ----------------------------------------------------
        # Widely separated information columns
        # ----------------------------------------------------

        ax_monitor.text(
            0.00,
            0.95,
            left_text,
            transform=ax_monitor.transAxes,
            ha='left',
            va='top',
            fontsize=9.0,
            family='monospace',
            linespacing=1.25
        )


        ax_monitor.text(
            0.66,
            0.95,
            right_text,
            transform=ax_monitor.transAxes,
            ha='left',
            va='top',
            fontsize=9.0,
            family='monospace',
            linespacing=1.25
        )


        fig.suptitle(
            'Large-Scale Overflow Dynamics in a Second-Order Recursive System',
            fontsize=13
        )


        plt.subplots_adjust(
            left=0.06,
            right=0.985,
            top=0.91,
            bottom=0.13
        )


        plt.show()

        plt.close(fig)


    # ========================================================
    # MODE 2 — PARAMETER-SPACE VIEW
    # ========================================================

    else:

        fig = plt.figure(
            figsize=(12.8, 5.7)
        )


        grid = fig.add_gridspec(
            1,
            2,
            width_ratios=[1.35, 0.82],
            wspace=0.28
        )


        ax_map = fig.add_subplot(
            grid[0, 0]
        )


        ax_info = fig.add_subplot(
            grid[0, 1]
        )


        # ====================================================
        # LINEAR STABILITY TRIANGLE
        # ====================================================

        stability_vertices = np.array([
            [-1.0, -2.0],
            [-1.0,  2.0],
            [ 1.0,  0.0]
        ])


        stability_patch = Polygon(
            stability_vertices,
            closed=True,
            facecolor='tab:blue',
            edgecolor='tab:blue',
            alpha=0.10,
            linewidth=2.0,
            label='Linear stability region'
        )


        ax_map.add_patch(
            stability_patch
        )


        # ====================================================
        # STRONGER SAFE REGION
        # ====================================================

        safe_vertices = np.array([
            [-1.0,  0.0],
            [ 0.0,  1.0],
            [ 1.0,  0.0],
            [ 0.0, -1.0]
        ])


        safe_patch = Polygon(
            safe_vertices,
            closed=True,
            facecolor='tab:green',
            edgecolor='tab:green',
            alpha=0.18,
            linewidth=2.0,
            label=r'$|\alpha|+|\beta|<1$'
        )


        ax_map.add_patch(
            safe_patch
        )


        # ====================================================
        # SELECTED PARAMETER POINT
        # ====================================================

        if stable and safe:

            point_color = 'tab:green'


        elif stable:

            point_color = 'tab:orange'


        else:

            point_color = 'tab:red'


        ax_map.plot(
            beta,
            alpha,
            'o',
            color=point_color,
            markersize=12,
            markeredgecolor='black',
            markeredgewidth=0.8,
            label='Selected system'
        )


        ax_map.axvline(
            0.0,
            color='0.5',
            linewidth=0.7
        )


        ax_map.axhline(
            0.0,
            color='0.5',
            linewidth=0.7
        )


        ax_map.set_xlim(
            -1.15,
            1.15
        )


        ax_map.set_ylim(
            -2.20,
            2.20
        )


        ax_map.set_xlabel(
            r'$\beta$',
            fontsize=12
        )


        ax_map.set_ylabel(
            r'$\alpha$',
            fontsize=12
        )


        ax_map.set_title(
            'Parameter-Space Stability Map',
            fontsize=13
        )


        ax_map.grid(
            True,
            linestyle=':',
            alpha=0.28
        )


        ax_map.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.11),
            ncol=3,
            frameon=False,
            fontsize=9
        )


        # ====================================================
        # PARAMETER STATUS PANEL
        # ====================================================

        ax_info.axis(
            'off'
        )


        if stable and safe:

            status_title = (
                'SAFE REGION'
            )


            status_body = (
                'Linear stability satisfied.\n'
                'No-overflow-oscillation\n'
                'condition satisfied.'
            )


        elif stable and not safe:

            status_title = (
                'CAUTION REGION'
            )


            status_body = (
                'Linear stability satisfied.\n\n'
                'However,\n'
                '|alpha| + |beta| < 1\n'
                'is NOT satisfied.\n\n'
                'Overflow oscillations\n'
                'may be possible.'
            )


        else:

            status_title = (
                'LINEARLY UNSTABLE'
            )


            status_body = (
                'At least one pole lies\n'
                'on or outside the unit circle.\n\n'
                'The linear system itself\n'
                'is not asymptotically stable.'
            )


        info_text = (
            f'PARAMETER MONITOR\n'
            f'────────────────────────\n'
            f'alpha          : {alpha:+.4f}\n'
            f'beta           : {beta:+.4f}\n'
            f'|alpha|+|beta| : {abs(alpha) + abs(beta):.4f}\n\n'
            f'POLES\n'
            f'p1 : {poles[0].real:+.4f}{poles[0].imag:+.4f}j\n'
            f'p2 : {poles[1].real:+.4f}{poles[1].imag:+.4f}j\n'
            f'|p1| : {abs(poles[0]):.4f}\n'
            f'|p2| : {abs(poles[1]):.4f}\n\n'
            f'{status_title}\n'
            f'────────────────────────\n'
            f'{status_body}'
        )


        ax_info.text(
            0.04,
            0.94,
            info_text,
            transform=ax_info.transAxes,
            ha='left',
            va='top',
            fontsize=10.0,
            family='monospace',
            linespacing=1.30,
            bbox=dict(
                boxstyle='round,pad=0.65',
                facecolor='white',
                edgecolor='0.4',
                alpha=0.96
            )
        )


        fig.suptitle(
            'Linear Stability versus Overflow-Oscillation Safety',
            fontsize=13
        )


        plt.subplots_adjust(
            left=0.07,
            right=0.97,
            top=0.90,
            bottom=0.18
        )


        plt.show()

        plt.close(fig)


# ============================================================
# CONTROLS
# ============================================================


# ------------------------------------------------------------
# View selector
#
# Short visible names are used so that the ToggleButtons remain
# completely readable. Their full meaning is explained in the
# documentation at the top of the notebook.
# ------------------------------------------------------------

view_buttons = ToggleButtons(
    options=[
        ('State plane', 'State-plane experiment'),
        ('Parameter plane', 'Parameter-space view')
    ],
    value='State-plane experiment',
    description='View:',
    style={
        'description_width': '45px'
    },
    layout=Layout(width='420px')
)


# ------------------------------------------------------------
# System coefficients
# ------------------------------------------------------------

alpha_slider = FloatSlider(
    value=1.50,
    min=-2.00,
    max=2.00,
    step=0.05,
    description='alpha:',
    continuous_update=True,
    readout_format='.2f',
    style={
        'description_width': '55px'
    },
    layout=Layout(width='360px')
)


beta_slider = FloatSlider(
    value=-0.80,
    min=-1.00,
    max=1.00,
    step=0.05,
    description='beta:',
    continuous_update=True,
    readout_format='.2f',
    style={
        'description_width': '50px'
    },
    layout=Layout(width='350px')
)


# ------------------------------------------------------------
# Initial conditions
# ------------------------------------------------------------

q10_slider = FloatSlider(
    value=-0.90,
    min=-1.00,
    max=1.00,
    step=0.05,
    description='q1[0]:',
    continuous_update=True,
    readout_format='.2f',
    style={
        'description_width': '55px'
    },
    layout=Layout(width='300px')
)


q20_slider = FloatSlider(
    value=0.90,
    min=-1.00,
    max=1.00,
    step=0.05,
    description='q2[0]:',
    continuous_update=True,
    readout_format='.2f',
    style={
        'description_width': '55px'
    },
    layout=Layout(width='300px')
)


# ------------------------------------------------------------
# Overflow recovery rule
# ------------------------------------------------------------

recovery_buttons = ToggleButtons(
    options=[
        'Saturation',
        'Zeroing',
        "Two's-complement wrap"
    ],
    value='Saturation',
    description='Recovery:',
    style={
        'description_width': '70px'
    },
    layout=Layout(width='535px')
)


# ------------------------------------------------------------
# Number of iterations
# ------------------------------------------------------------

iterations_slider = IntSlider(
    value=30,
    min=10,
    max=80,
    step=5,
    description='Iterations:',
    continuous_update=True,
    style={
        'description_width': '75px'
    },
    layout=Layout(width='320px')
)


# ------------------------------------------------------------
# Time step
# ------------------------------------------------------------

time_slider = IntSlider(
    value=1,
    min=1,
    max=30,
    step=1,
    description='Time step n:',
    continuous_update=True,
    style={
        'description_width': '85px'
    },
    layout=Layout(width='600px')
)


# ------------------------------------------------------------
# Play control
# ------------------------------------------------------------

play_control = Play(
    value=1,
    min=1,
    max=30,
    step=1,
    interval=800,
    description='Play',
    disabled=False,
    layout=Layout(width='120px')
)


# ------------------------------------------------------------
# Synchronize Play and Time step
# ------------------------------------------------------------

jslink(
    (play_control, 'value'),
    (time_slider, 'value')
)


# ============================================================
# DYNAMIC CONTROL BEHAVIOR
# ============================================================


# ------------------------------------------------------------
# Controls locked during automatic animation
# ------------------------------------------------------------

animation_lock_controls = [
    view_buttons,
    alpha_slider,
    beta_slider,
    q10_slider,
    q20_slider,
    recovery_buttons,
    iterations_slider,
    time_slider
]


# ------------------------------------------------------------
# Enable or disable mode-specific controls
# ------------------------------------------------------------

def update_view_controls(change=None):

    parameter_mode = view_buttons.value == 'Parameter-space view'


    q10_slider.disabled = parameter_mode

    q20_slider.disabled = parameter_mode

    recovery_buttons.disabled = parameter_mode

    iterations_slider.disabled = parameter_mode

    time_slider.disabled = parameter_mode

    play_control.disabled = parameter_mode


# ------------------------------------------------------------
# Lock all model controls during automatic playback
# ------------------------------------------------------------

def set_animation_lock(locked):

    for widget in animation_lock_controls:

        widget.disabled = locked


    if not locked:

        update_view_controls()


def update_animation_lock(change):

    set_animation_lock(
        bool(change['new'])
    )


# ------------------------------------------------------------
# Detect the Play-state trait supported by ipywidgets
# ------------------------------------------------------------

play_traits = play_control.traits()


if 'playing' in play_traits:

    play_state_trait = 'playing'


elif '_playing' in play_traits:

    play_state_trait = '_playing'


else:

    play_state_trait = None


if play_state_trait is not None:

    play_control.observe(
        update_animation_lock,
        names=play_state_trait
    )


# ------------------------------------------------------------
# React to mode changes
# ------------------------------------------------------------

view_buttons.observe(
    update_view_controls,
    names='value'
)


# ------------------------------------------------------------
# Synchronize iteration count and animation range
# ------------------------------------------------------------

def update_iteration_range(change):

    new_maximum = change['new']

    time_slider.max = new_maximum

    play_control.max = new_maximum


    if time_slider.value > new_maximum:

        time_slider.value = new_maximum


    if play_control.value > new_maximum:

        play_control.value = new_maximum


iterations_slider.observe(
    update_iteration_range,
    names='value'
)


# ------------------------------------------------------------
# Changing a state-plane parameter starts a new experiment
# ------------------------------------------------------------

def restart_time(change):

    if view_buttons.value == 'State-plane experiment':

        time_slider.value = 1

        play_control.value = 1


alpha_slider.observe(
    restart_time,
    names='value'
)


beta_slider.observe(
    restart_time,
    names='value'
)


q10_slider.observe(
    restart_time,
    names='value'
)


q20_slider.observe(
    restart_time,
    names='value'
)


recovery_buttons.observe(
    restart_time,
    names='value'
)


# ------------------------------------------------------------
# Initial state of controls
# ------------------------------------------------------------

update_view_controls()


# ============================================================
# CONTROL LAYOUT
# ============================================================

controls_row_1 = HBox(
    [
        view_buttons,
        alpha_slider,
        beta_slider
    ],
    layout=Layout(
        width='960px',
        justify_content='space-between',
        align_items='center'
    )
)


controls_row_2 = HBox(
    [
        q10_slider,
        q20_slider,
        iterations_slider
    ],
    layout=Layout(
        width='920px',
        justify_content='space-between',
        align_items='center'
    )
)


controls_row_3 = HBox(
    [
        recovery_buttons,
        play_control,
        time_slider
    ],
    layout=Layout(
        width='960px',
        justify_content='space-between',
        align_items='center'
    )
)


controls_box = VBox(
    [
        HTML("<div class='olc-controls-title'>Experiment controls</div>"),
        controls_row_1,
        controls_row_2,
        controls_row_3
    ],
    layout=Layout(
        width='960px',
        border='1px solid #d4d9df',
        padding='6px 8px',
        overflow='visible'
    )
)


# ============================================================
# INTERACTIVE OBJECT
# ============================================================

widget_plot = interactive(
    plot_overflow_lab,
    view=view_buttons,
    alpha=alpha_slider,
    beta=beta_slider,
    q10=q10_slider,
    q20=q20_slider,
    recovery=recovery_buttons,
    iterations=iterations_slider,
    time_step=time_slider
)


plot_output = widget_plot.children[-1]


plot_output.layout = Layout(
    width='auto',
    overflow='visible'
)


# ============================================================
# FINAL NOTEBOOK LAYOUT
# ============================================================

main_layout = VBox(
    [
        header_html,
        controls_box,
        plot_output
    ],
    layout=Layout(
        width='960px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ============================================================
# DISPLAY
# ============================================================

display(style_html)

display(main_layout)